# AI-Orb · Synapses Stimme auf Colab (Kartoffelbox)

Deutsches Sprachmodell **Kartoffelbox** (Chatterbox, auf Deutsch nachtrainiert) mit eingebauter deutscher Stimme (oder auf Wunsch deiner eigenen), Ausdrucksstärke und Lauten wie Lachen. Der Orb holt sich hier jeden Satz ab; ist das Notebook aus, spricht er automatisch mit seiner bisherigen Stimme weiter.

### Einmalig vorbereiten (ca. 5 Minuten)
1. **Hugging Face:** Konto anlegen, auf <https://huggingface.co/SebastianBodza/Kartoffelbox-v0.1> die Bedingungen akzeptieren, unter *Settings → Access Tokens* einen Token (Typ *Read*) erzeugen.
2. **Tailscale:** <https://login.tailscale.com/admin/settings/keys> → *Generate auth key* → **Reusable** und **Ephemeral** anhaken.
3. Hier in Colab links auf das **Schlüssel-Symbol (Secrets)**: `HF_TOKEN` und `TS_AUTHKEY` anlegen und jeweils *Notebook access* einschalten.
4. **Laufzeit → Laufzeittyp ändern → T4 GPU.**

### Jedes Mal
**Laufzeit → Alle ausführen.** Eine Aufnahme brauchst du nicht – es spricht die eingebaute Stimme von Kartoffelbox. (Wer später doch eine eigene Stimme will: in Zelle 2 `EIGENE_STIMME` anhaken.) **Den Tab offen lassen**: Colab Gratis beendet Sitzungen nach einigen Stunden oder wenn der Tab lange unbeachtet ist.

In [ ]:
# 1/4 · GPU prüfen und Sprachmodell in eine eigene Umgebung installieren (beim ersten Mal 3–5 Minuten)
# 19.11.: eigene Umgebung (/content/stimme-env) – Colabs eigenes torch bleibt
# unberührt, damit gibt es keine Versionskonflikte (torchvision, transformers) mehr.
import subprocess, sys, os
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("Keine GPU. Laufzeit → Laufzeittyp ändern → T4 GPU, dann neu starten.")
print("GPU:", gpu)

STIMME_PY = "/content/stimme-env/bin/python"
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False, stdout=subprocess.DEVNULL)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not os.path.exists(STIMME_PY):
    subprocess.check_call(["uv", "venv", "-q", "-p", "3.12", "/content/stimme-env"])
erg = subprocess.run(["uv", "pip", "install", "-q", "--python", STIMME_PY,
                      "chatterbox-tts==0.1.7", "fastapi>=0.115", "uvicorn>=0.30", "soundfile",
                      "setuptools<81"]  # pkg_resources fuer das Wasserzeichen (19.11.),
                     capture_output=True, text=True)
if erg.returncode != 0:
    print(erg.stdout[-2000:], erg.stderr[-3000:])
    raise SystemExit("Installation fehlgeschlagen – Meldung oben.")
test = subprocess.run([STIMME_PY, "-c", "import torch, chatterbox.tts; print(torch.__version__, torch.cuda.is_available())"],
                      capture_output=True, text=True)
print(test.stdout.strip() or test.stderr[-2000:])
if "True" not in test.stdout:
    raise SystemExit("Das Sprachmodell sieht die GPU nicht – Meldung oben.")
print("✓ Pakete bereit")


In [ ]:
# 2/4 · Stimme: eingebaut (Vorgabe) – oder optional eine eigene Aufnahme
import subprocess
from pathlib import Path

EIGENE_STIMME = False  # @param {type:"boolean"}
NEU_AUFNEHMEN = False  # @param {type:"boolean"}

REFERENZ = None
if EIGENE_STIMME:
    from google.colab import drive, files
    drive.mount("/content/drive")
    ORDNER = Path("/content/drive/MyDrive/AI-Orb-Stimme")
    ORDNER.mkdir(parents=True, exist_ok=True)
    REFERENZ = ORDNER / "referenz.wav"
    if NEU_AUFNEHMEN or not REFERENZ.exists():
        print("Lade jetzt deine Aufnahme hoch (WAV, MP3 oder M4A; 10–30 Sekunden, ruhig, ohne Musik).")
        hochgeladen = files.upload()
        if len(hochgeladen) != 1:
            raise SystemExit("Bitte genau eine Datei hochladen.")
        quelle = Path(next(iter(hochgeladen)))
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(quelle),
                        "-t", "25", "-ac", "1", "-ar", "24000",
                        "-af", "silenceremove=start_periods=1:start_threshold=-45dB",
                        str(REFERENZ)], check=True)
        print("✓ Eigene Stimme gespeichert:", REFERENZ)
    else:
        print("✓ Eigene Stimme aus Google Drive:", REFERENZ)
else:
    print("✓ Es spricht die eingebaute Stimme von Kartoffelbox – keine Aufnahme nötig.")

In [ ]:
%%writefile /content/kartoffel_tts_server.py
"""AI-Orb Sprachserver: Kartoffelbox (deutsches Chatterbox) auf einer Colab-GPU.

17.11.2026 - Marc will eine eigene Stimme mit einem eigenen Sprachmodell,
das auf Google Colab (Gratis, T4) laeuft.

Was dieser Server tut
---------------------
* Laedt **Kartoffelbox** (SebastianBodza/Kartoffelbox-v0.1, Stand v0.2): ein
  auf ~600k deutschen Saetzen nachtrainiertes Chatterbox mit einstellbarer
  Ausdrucksstaerke und Stimmuebernahme.
* Uebernimmt die Stimme aus Marcs Referenzaufnahme (einmal vorbereitet,
  danach fuer jeden Satz wiederverwendet - das spart Zeit je Satz).
* Antwortet auf ``POST /tts`` mit einer WAV-Datei - genau das, was
  ``backend/tts/remote_tts_engine.py`` im Orb erwartet.
* Uebersetzt die Laut-Tags des Orb in die Laute, die Kartoffelbox kennt:
  ``<lachen>`` -> ``<haha>``, ``<schmunzeln>`` -> ``<chuckle>``,
  ``<hmm>`` -> ``<hmm_neugierig>`` usw. (Vorschau-Funktion des Modells:
  klingt mal besser, mal schlechter.)
* Die Gefuehlslage des Orb steuert die Ausdrucksstaerke, das Tempo die
  CFG-Gewichtung (weniger Gewichtung = ruhigeres, bedaechtigeres Sprechen).

Gestartet wird er vom Colab-Notebook als eigener Prozess - damit stoeren
die Paketversionen des Modells nicht den Notebook-Kern.

Umgebung:
    ORB_REFERENZ   Pfad zur Referenzaufnahme (WAV, 24 kHz mono)
    HF_TOKEN       Hugging-Face-Zugang (Kartoffelbox ist zugangsbeschraenkt)
    ORB_TOKEN      optional: verlangt "Authorization: Bearer <token>"
    ORB_PORT       Port (Vorgabe 8800)
"""

from __future__ import annotations

import io
import os
import re
import threading
import time

import soundfile as sf
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Response
from huggingface_hub import snapshot_download
from pydantic import BaseModel, Field

MODELL_REPO = "SebastianBodza/Kartoffelbox-v0.1"
PORT = int(os.environ.get("ORB_PORT", "8800"))
REFERENZ = os.environ.get("ORB_REFERENZ", "").strip()
TOKEN = os.environ.get("ORB_TOKEN", "").strip()

#: Laute des Orb -> Laute, die Kartoffelbox v0.2 gelernt hat.
LAUTE = {
    "lachen": "<haha>",
    "schmunzeln": "<chuckle>",
    "hmm": "<hmm_neugierig>",
    "oh": "<ohhh>",
    "wow": "<wow>",
    "ach": "<aaach>",
    "seufzen": "<aaach>",
}
_LAUT_TAG = re.compile(r"<\s*(" + "|".join(LAUTE) + r")\s*/?\s*>", re.IGNORECASE)
#: Alles andere, was nach Markup aussieht, wird nie gesprochen.
_REST_TAG = re.compile(r"</?\s*[a-zA-Z_]+[^<>]{0,40}>")

#: Gefuehlslage des Orb -> Ausdrucksstaerke (Chatterbox: 0.25 ruhig ... 1.0 dramatisch).
AUSDRUCK = {
    "neutral": 0.50, "freundlich": 0.55, "aufmerksam": 0.50, "nachdenklich": 0.42,
    "erfreut": 0.68, "bedauernd": 0.45, "dringlich": 0.62, "leise": 0.38,
    "friendly": 0.55, "curious": 0.58, "thinking": 0.42, "happy": 0.68,
    "calm": 0.40, "warning": 0.62, "error": 0.45,
}

print("[stimme] Lade Kartoffelbox ...", flush=True)
import perth  # noqa: E402

# Das Wasserzeichen braucht pkg_resources (setuptools). Fehlt es, ist
# PerthImplicitWatermarker None und Chatterbox stuerzt beim Laden ab (19.11.2026).
if getattr(perth, "PerthImplicitWatermarker", None) is None:
    print("[stimme] Wasserzeichen nicht verfuegbar - spreche ohne.", flush=True)
    perth.PerthImplicitWatermarker = perth.DummyWatermarker
from chatterbox.tts import ChatterboxTTS  # noqa: E402 - erst nach der Meldung
from safetensors.torch import load_file  # noqa: E402

GERAET = "cuda" if torch.cuda.is_available() else "cpu"
HF_TOKEN = os.environ.get("HF_TOKEN") or None
VOLLSTAENDIG = ("t3_cfg.safetensors", "ve.safetensors", "s3gen.safetensors", "tokenizer.json")


def modell_laden() -> ChatterboxTTS:
    """Laedt Kartoffelbox auf zwei Wegen - der zweite ist der aus der Modellkarte.

    1. Das Repo enthaelt seit v0.2 alle Dateien im Chatterbox-Aufbau ->
       ``from_local``.
    2. Fehlt etwas oder passt es nicht: das Grundmodell von Resemble laden
       und nur das deutsche T3 hineinsetzen (so steht es auf der Modellkarte).
    """
    try:
        ordner = snapshot_download(
            MODELL_REPO, token=HF_TOKEN,
            allow_patterns=["*.safetensors", "*.safetensor", "*.json", "*.pt"],
        )
    except Exception as fehler:  # noqa: BLE001
        text = str(fehler)
        if "401" in text or "403" in text or "gated" in text.lower():
            raise SystemExit(
                "[stimme] FEHLER: Kein Zugriff auf Kartoffelbox. Auf "
                "https://huggingface.co/SebastianBodza/Kartoffelbox-v0.1 eingeloggt die "
                "Bedingungen akzeptieren und pruefen, dass das Secret HF_TOKEN zu diesem Konto gehoert."
            ) from fehler
        raise
    vorhanden = set(os.listdir(ordner))
    print(f"[stimme] Dateien im Modell: {sorted(vorhanden)}", flush=True)
    if all(name in vorhanden for name in VOLLSTAENDIG):
        try:
            m = ChatterboxTTS.from_local(ordner, GERAET)
            print("[stimme] Kartoffelbox vollstaendig geladen.", flush=True)
            return m
        except Exception as fehler:  # noqa: BLE001
            print(f"[stimme] Vollstaendiges Laden ging nicht ({fehler}) - nehme den Weg der Modellkarte.", flush=True)
    t3_datei = next((n for n in ("t3_cfg.safetensors", "t3_cfg.safetensor") if n in vorhanden), None)
    if t3_datei is None:
        raise SystemExit("[stimme] FEHLER: t3_cfg.safetensors fehlt im Kartoffelbox-Repo.")
    m = ChatterboxTTS.from_pretrained(device=GERAET)
    zustand = load_file(os.path.join(ordner, t3_datei), device="cpu")
    if "model" in zustand:
        zustand = zustand["model"][0]
    m.t3.load_state_dict(zustand)
    m.t3.to(GERAET).eval()
    print("[stimme] Grundmodell geladen, deutsches T3 eingesetzt.", flush=True)
    return m


modell = modell_laden()

# --- Schneller rechnen (19.11.2026) -------------------------------------
# Marc: zu lange Sprechpausen. Zwei Stellschrauben, beide abschaltbar:
#  * halbe Genauigkeit (fp16) per autocast - die T4 rechnet damit auf ihren
#    Tensor-Kernen ein Vielfaches schneller als in fp32;
#  * weniger Schritte im Klangdecoder (S3Gen-Flow, Vorgabe 10).
SCHNELL = os.environ.get("ORB_SCHNELL", "1") != "0" and GERAET == "cuda"

# Hoechstlaenge je Satz (19.11.2026): Chatterbox findet manchmal das Satzende
# nicht und erzeugt bis zu 1000 Tonschritte (~40 s Gebrabbel). Waehrenddessen
# wartete der Orb, lief in seine Frist und sprach 47 s lang nichts. Etwa
# 25 Schritte sind eine Sekunde; erlaubt ist gut das Doppelte der zu
# erwartenden Sprechdauer.
_GRENZE = {"schritte": 1000}
_t3_inferenz = modell.t3.inference


def _t3_begrenzt(*args, **kwargs):
    kwargs["max_new_tokens"] = min(int(kwargs.get("max_new_tokens") or 1000), _GRENZE["schritte"])
    return _t3_inferenz(*args, **kwargs)


modell.t3.inference = _t3_begrenzt


def schritte_fuer(text: str) -> int:
    sekunden = len(text) / 11.0 + 2.0      # ~11 Zeichen je Sekunde, plus Luft
    return max(60, min(1000, int(25 * sekunden * 1.3)))
CFM_SCHRITTE = int(os.environ.get("ORB_CFM_SCHRITTE", "6"))
if CFM_SCHRITTE > 0:
    _s3_inferenz = modell.s3gen.inference

    def _s3_kuerzer(*args, **kwargs):
        kwargs.setdefault("n_cfm_timesteps", CFM_SCHRITTE)
        return _s3_inferenz(*args, **kwargs)

    modell.s3gen.inference = _s3_kuerzer
if REFERENZ and os.path.exists(REFERENZ):
    modell.prepare_conditionals(REFERENZ, exaggeration=0.5)
    print(f"[stimme] Referenzstimme uebernommen: {REFERENZ}", flush=True)
else:
    print("[stimme] Keine Referenz - es spricht die eingebaute Stimme.", flush=True)

sperre = threading.Lock()
app = FastAPI(title="AI-Orb Stimme (Kartoffelbox)", docs_url=None, redoc_url=None)


class Auftrag(BaseModel):
    text: str = Field(min_length=1, max_length=800)
    speaker: int = 0
    speed: float = Field(default=1.0, ge=0.5, le=2.0)
    emotion: str = Field(default="neutral", max_length=32)


def text_vorbereiten(text: str) -> str:
    text = _LAUT_TAG.sub(lambda m: f" {LAUTE[m.group(1).lower()]} ", text)
    # Laut-Tags von Kartoffelbox selbst sollen stehen bleiben, alles andere weg.
    erlaubt = set(LAUTE.values())
    text = _REST_TAG.sub(lambda m: m.group(0) if m.group(0) in erlaubt else " ", text)
    return " ".join(text.split())


def sprechen(text: str, speed: float, emotion: str) -> tuple[bytes, float]:
    ausdruck = AUSDRUCK.get(emotion.strip().lower(), 0.5)
    if speed < 0.93:
        cfg = 0.25      # bedaechtiger
    elif speed > 1.06:
        cfg = 0.45      # zuegiger
    else:
        cfg = 0.32
    global SCHNELL
    start = time.perf_counter()
    with sperre, torch.inference_mode():
        _GRENZE["schritte"] = schritte_fuer(text)
        wav = None
        if SCHNELL:
            try:
                with torch.autocast("cuda", dtype=torch.float16):
                    wav = modell.generate(text, exaggeration=ausdruck, cfg_weight=cfg, temperature=0.65)
                if not torch.isfinite(wav).all() or float(wav.abs().max()) < 1e-4:
                    raise ValueError("fp16 lieferte keinen brauchbaren Ton")
            except Exception as fehler:  # noqa: BLE001 - dann eben in voller Genauigkeit
                print(f"[stimme] fp16 geht nicht ({fehler}) - rechne in fp32 weiter.", flush=True)
                SCHNELL = False
                wav = None
        if wav is None:
            wav = modell.generate(text, exaggeration=ausdruck, cfg_weight=cfg, temperature=0.65)
    daten = wav.squeeze().float().cpu().numpy()
    puffer = io.BytesIO()
    sf.write(puffer, daten, modell.sr, format="WAV", subtype="PCM_16")
    dauer = time.perf_counter() - start
    hoerdauer = len(daten) / modell.sr
    print(f"[stimme] {hoerdauer:.1f} s Ton in {dauer:.2f} s (Faktor {dauer / max(hoerdauer, 0.1):.2f})", flush=True)
    return puffer.getvalue(), dauer


@app.get("/health")
def health() -> dict:
    return {
        "ok": True,
        "modell": "Kartoffelbox (deutsches Chatterbox)",
        "geraet": torch.cuda.get_device_name(0) if GERAET == "cuda" else "cpu",
        "referenz": bool(REFERENZ and os.path.exists(REFERENZ)),
        "schnell": SCHNELL,
        "cfm_schritte": CFM_SCHRITTE,
    }


@app.post("/tts")
def tts(auftrag: Auftrag, authorization: str | None = Header(default=None)) -> Response:
    if TOKEN and authorization != f"Bearer {TOKEN}":
        raise HTTPException(status_code=401, detail="Falscher Zugangstoken.")
    text = text_vorbereiten(auftrag.text)
    if not text:
        raise HTTPException(status_code=400, detail="Kein Text.")
    try:
        daten, dauer = sprechen(text, auftrag.speed, auftrag.emotion)
    except Exception as fehler:  # noqa: BLE001 - als Meldung an den Orb, nicht als Absturz.
        raise HTTPException(status_code=500, detail=f"Erzeugung fehlgeschlagen: {fehler}") from fehler
    print(f"[stimme] {dauer:.2f} s fuer {len(text)} Zeichen: {text[:60]}", flush=True)
    return Response(content=daten, media_type="audio/wav")


if __name__ == "__main__":
    # Aufwaermen: der erste Satz kompiliert Kernels und dauert laenger.
    sprechen("Hallo, ich bin bereit.", 1.0, "freundlich")
    sprechen("Das ist ein zweiter, etwas laengerer Satz zum Messen der Geschwindigkeit.", 1.0, "neutral")
    print(f"[stimme] Bereit auf Port {PORT}.", flush=True)
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")


In [ ]:
# 3/4 · Sprachserver starten und Hörprobe abspielen
import os, subprocess, sys, time, json, urllib.request
from google.colab import userdata
from IPython.display import Audio, display

def secret(name):
    try:
        # Nur das erste Stueck: ein mehrfach eingefuegter Schluessel (19.11.) wird so trotzdem gelesen.
        teile = (userdata.get(name) or "").split()
        return teile[0] if teile else ""
    except Exception:
        return ""

umgebung = dict(os.environ, HF_TOKEN=secret("HF_TOKEN"), ORB_REFERENZ=str(REFERENZ) if REFERENZ else "", ORB_PORT="8800")
if not umgebung["HF_TOKEN"]:
    raise SystemExit("Secret HF_TOKEN fehlt oder 'Notebook access' ist aus (Schlüssel-Symbol links).")

# Token und Freigabe vorab prüfen – klare Meldung statt Absturz im Server.
def hf(pfad):
    anfrage = urllib.request.Request("https://huggingface.co" + pfad,
                                     headers={"Authorization": "Bearer " + umgebung["HF_TOKEN"]})
    return urllib.request.urlopen(anfrage, timeout=20)
try:
    konto = json.loads(hf("/api/whoami-v2").read()).get("name")
    print("✓ Hugging Face Konto:", konto)
except Exception:
    raise SystemExit("HF_TOKEN ist ungültig. Unter huggingface.co → Settings → Access Tokens neu erzeugen (Typ Read).")
import urllib.error
try:
    hf("/api/models/SebastianBodza/Kartoffelbox-v0.1/auth-check")
    print("✓ Zugriff auf Kartoffelbox freigegeben")
except urllib.error.HTTPError as f:
    if f.code not in (401, 403):
        print("(Freigabe nicht prüfbar, Code", f.code, "– weiter)")
    else:
        raise SystemExit("Kein Zugriff auf Kartoffelbox: eingeloggt auf https://huggingface.co/SebastianBodza/Kartoffelbox-v0.1 "
                         "die Bedingungen akzeptieren ('Agree and access repository'), dann diese Zelle neu starten.")
except Exception as f:
    print("(Freigabe nicht prüfbar:", f, "– weiter)")

try:
    server.kill()
except Exception:
    pass
protokoll = open("/content/stimme.log", "w")
server = subprocess.Popen([STIMME_PY, "/content/kartoffel_tts_server.py"],
                          stdout=protokoll, stderr=subprocess.STDOUT, env=umgebung, start_new_session=True)

print("Modell wird geladen (beim ersten Mal 2–4 Minuten) ...")
bereit = False
for _ in range(450):
    time.sleep(2)
    if server.poll() is not None:
        print(open("/content/stimme.log").read()[-3000:])
        raise SystemExit("Der Sprachserver ist abgestürzt – Protokoll oben.")
    try:
        print("✓", json.loads(urllib.request.urlopen("http://127.0.0.1:8800/health", timeout=2).read()))
        bereit = True
        break
    except Exception:
        pass
if not bereit:
    print(open("/content/stimme.log").read()[-3000:])
    raise SystemExit("Der Sprachserver wurde in 15 Minuten nicht fertig – Protokoll oben.")

probe = json.dumps({"text": "Hallo Marc. <schmunzeln> Schön, dass du da bist. Ich klinge jetzt so.",
                    "emotion": "freundlich"}).encode()
anfrage = urllib.request.Request("http://127.0.0.1:8800/tts", data=probe,
                                 headers={"Content-Type": "application/json"})
start = time.time()
ton = urllib.request.urlopen(anfrage, timeout=120).read()
print(f"Hörprobe in {time.time() - start:.1f} s erzeugt:")
display(Audio(ton, rate=24000))

In [ ]:
# 4/4 · Feste Adresse im Tailnet (orb-stimme) – diese Zelle läuft weiter, solange die Stimme gebraucht wird
import subprocess, time, re, json, urllib.request, os, sys
seite = urllib.request.urlopen("https://pkgs.tailscale.com/stable/", timeout=30).read().decode()
datei = max(set(re.findall(r"tailscale_[\d.]+_amd64\.tgz", seite)), key=lambda d: [int(x) for x in re.findall(r"\d+", d)])
print("1/4 Lade", datei); urllib.request.urlretrieve("https://pkgs.tailscale.com/stable/" + datei, "/content/ts.tgz")
subprocess.run("tar xzf /content/ts.tgz -C /content && cp /content/tailscale_*_amd64/tailscale* /usr/local/bin/", shell=True, check=True)
print("2/4 Starte tailscaled"); subprocess.run(["pkill", "-f", "tailscaled"]); time.sleep(1)
subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--state=mem:", "--socket=/tmp/ts.sock"], stdout=open("/content/tailscaled.log", "w"), stderr=subprocess.STDOUT, start_new_session=True); time.sleep(5)
print("3/4 Melde an")
oben = subprocess.run(["tailscale", "--socket=/tmp/ts.sock", "up", "--authkey=" + secret("TS_AUTHKEY"), "--hostname=orb-stimme", "--accept-dns=false", "--reset"], capture_output=True, text=True, timeout=60)
print(oben.stdout[-500:], oben.stderr[-500:])
name = json.loads(subprocess.run(["tailscale", "--socket=/tmp/ts.sock", "status", "--json"], capture_output=True, text=True).stdout)["Self"]["DNSName"].rstrip(".")
print("4/4 ✓ Tailnet-Adresse:", f"http://{name}:8800", "– Tab offen lassen.")
while True:
    if server.poll() is not None:
        server = subprocess.Popen([STIMME_PY, "/content/kartoffel_tts_server.py"], stdout=open("/content/stimme.log", "a"), stderr=subprocess.STDOUT, env=umgebung, start_new_session=True)
    print(time.strftime("%H:%M"), "läuft ·", sum(" s fuer " in z for z in open("/content/stimme.log")), "Sätze", end="\r"); time.sleep(60)
